# Validate runtime context for a synthetic HR assistant

Keep authenticated identity separate from user-provided tool arguments, compute leave balances from synthetic events, verify tenant isolation, and construct an optional Orchestrate run with explicit context.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** bind a leave-summary tool to authenticated tenant and subject scope; test date bounds, identity overrides, and isolated records; separate client-writable runtime context from server-enforced authorization.

**Prerequisites:** Understand Python dictionaries, dataclasses, exceptions, and date intervals. The local example uses only the standard library. The optional HTTPX request requires a deployed synthetic Orchestrate agent and approved endpoint/token; it verifies run acceptance only.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Represent identity and leave events separately

The invented employee identifiers contain no real personnel information. TrustedContext stands for context produced by authentication middleware; constructing this dataclass locally is not an implementation of JWT verification.


In [ ]:
import json
from dataclasses import dataclass
from datetime import date
from decimal import Decimal


@dataclass(frozen=True)
class TrustedContext:
    tenant: str
    subject: str
    scopes: frozenset[str]


EVENTS = [
    {
        "tenant": "tenant-amber",
        "subject": "employee-101",
        "day": "2030-02-04",
        "hours": "8",
        "status": "approved",
    },
    {
        "tenant": "tenant-amber",
        "subject": "employee-101",
        "day": "2030-02-05",
        "hours": "4",
        "status": "approved",
    },
    {
        "tenant": "tenant-amber",
        "subject": "employee-101",
        "day": "2030-02-06",
        "hours": "8",
        "status": "cancelled",
    },
    {
        "tenant": "tenant-amber",
        "subject": "employee-102",
        "day": "2030-02-04",
        "hours": "8",
        "status": "approved",
    },
    {
        "tenant": "tenant-indigo",
        "subject": "employee-101",
        "day": "2030-02-04",
        "hours": "16",
        "status": "approved",
    },
]
CONTEXT = TrustedContext("tenant-amber", "employee-101", frozenset({"leave:read:self"}))
assert len(EVENTS) == 5

## 2. Compute a scoped leave summary

The user can choose a date interval, not their identity or tenant. Exact argument matching blocks identity overrides. Decimal avoids floating-point drift in fractional-hour totals; cancelled events do not consume approved leave.


In [ ]:
def leave_summary(arguments, context):
    if set(arguments) != {"start_date", "end_date"}:
        raise PermissionError("Only date filters may be supplied")
    if "leave:read:self" not in context.scopes:
        raise PermissionError("Leave access is not authorized")
    start = date.fromisoformat(arguments["start_date"])
    end = date.fromisoformat(arguments["end_date"])
    if start > end or (end - start).days > 366:
        raise ValueError("Invalid or excessive date interval")
    selected = [
        e
        for e in EVENTS
        if (e["tenant"], e["subject"]) == (context.tenant, context.subject)
        and start <= date.fromisoformat(e["day"]) <= end
        and e["status"] == "approved"
    ]
    hours = sum((Decimal(e["hours"]) for e in selected), Decimal(0))
    return {
        "approved_hours": str(hours),
        "approved_events": len(selected),
        "date_range": {"start": start.isoformat(), "end": end.isoformat()},
    }


arguments = {"start_date": "2030-02-01", "end_date": "2030-02-28"}
summary = leave_summary(arguments, CONTEXT)
assert summary["approved_hours"] == "12" and summary["approved_events"] == 2
print(summary)

## 3. Test authorization, dates, and tenant isolation

Negative cases matter as much as the happy path. Another tenant may reuse the same subject identifier, so filtering only by subject would leak records. User text does not grant a new scope.


In [ ]:
other = TrustedContext("tenant-indigo", "employee-101", frozenset({"leave:read:self"}))
assert leave_summary(arguments, other)["approved_hours"] == "16"
assert (
    leave_summary({"start_date": "2030-03-01", "end_date": "2030-03-31"}, CONTEXT)["approved_hours"]
    == "0"
)
for bad, error, context in [
    (dict(arguments, subject="employee-102"), PermissionError, CONTEXT),
    ({"start_date": "2030-03-01", "end_date": "2030-02-01"}, ValueError, CONTEXT),
    ({"start_date": "invalid", "end_date": "2030-02-01"}, ValueError, CONTEXT),
    (
        arguments,
        PermissionError,
        TrustedContext(CONTEXT.tenant, CONTEXT.subject, frozenset()),
    ),
]:
    try:
        leave_summary(bad, context)
    except error:
        pass
    else:
        raise AssertionError("Invalid request accepted")
# Telemetry uses an allowlist; no identifiers, prompts, or bearer token are logged.
telemetry = {
    "tool": "leave_summary",
    "outcome": "success",
    "event_count": summary["approved_events"],
}
assert set(telemetry) == {"tool", "outcome", "event_count"}
print({"negative_cases_rejected": 4, "tenant_isolation": True, "telemetry": telemetry})

## 4. Build the runtime request without confusing context with authorization

The official RunOrchestrate request supports message, agent_id, context, and context_variables. These application fields are not cryptographic proof of identity. In production, the tool backend must enforce the authenticated principal independently; do not trust a client-writable employee ID merely because it appears in context.


In [ ]:
def runtime_payload(agent_id, question):
    return {
        "agent_id": agent_id,
        "message": {"role": "user", "content": question},
        "context": {"locale": "en-US", "purpose": "synthetic-evaluation"},
        "context_variables": {
            "requested_start_date": arguments["start_date"],
            "requested_end_date": arguments["end_date"],
        },
    }


request_preview = runtime_payload(
    "synthetic-agent-id",
    "How many approved leave hours are in the specified date interval?",
)
assert "subject" not in request_preview["context_variables"]
assert request_preview["message"]["role"] == "user"
print(json.dumps(request_preview, indent=2))

## 5. Optional: start an Orchestrate run

Set WXO_RUNS_URL to the authorized HTTPS /v1/orchestrate/runs endpoint, WXO_AGENT_ID to your deployed test agent, and WXO_BEARER_TOKEN through your environment. The documented response returns run/thread/task/message IDs. Receiving IDs means the run was accepted; completion and authorization still require separate observation.

Transport/protocol failures are reported without their original exception text, which can echo a malformed credential header. Numeric HTTP-status failures remain visible; receiving identifiers still does not prove completion.


In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    import os
    from urllib.parse import urlsplit

    import httpx

    endpoint = os.environ["WXO_RUNS_URL"]
    try:
        parsed = urlsplit(endpoint)
    except ValueError:
        raise ValueError("HTTPS endpoint without embedded credentials required") from None
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.username
        or parsed.password
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError("HTTPS endpoint without embedded credentials required")
    payload = runtime_payload(os.environ["WXO_AGENT_ID"], request_preview["message"]["content"])
    request_headers = {"Authorization": "Bearer " + os.environ["WXO_BEARER_TOKEN"]}
    try:
        with httpx.Client(timeout=90, follow_redirects=False, trust_env=False) as client:
            response = client.post(endpoint, headers=request_headers, json=payload)
    except Exception:
        # Header/protocol errors may contain credentials; omit their original text.
        raise RuntimeError("Service transport failed; no response was verified") from None
    if not 200 <= response.status_code < 300:
        raise RuntimeError(f"Service request failed with HTTP {response.status_code}")
    try:
        accepted = response.json()
    except ValueError:
        raise ValueError("Unexpected Orchestrate run response") from None
    required_ids = {"run_id", "thread_id", "task_id", "message_id"}
    if (
        not isinstance(accepted, dict)
        or not required_ids <= accepted.keys()
        or any(not isinstance(accepted[k], str) or not accepted[k] for k in required_ids)
    ):
        raise ValueError("Unexpected Orchestrate run response")
    print(
        {
            "accepted": True,
            "completion_verified": False,
            "backend_authorization_verified": False,
        }
    )
else:
    print("No live run started. All leave and authorization checks above used invented local data.")

## Expected local results

The selected fictional employee has 12 approved hours from two events. A different tenant using the same subject identifier has 16 hours, and an empty interval returns zero. Four invalid date/authorization cases are rejected.

**Try next:** test concurrent requests for two authenticated identities and verify that logs never capture raw context or credentials.


## Interpretation, limitations, and cleanup

The local checks establish the leave tool’s data-access contract, not Orchestrate authentication or context propagation. A deployment must verify JWT signature, issuer, audience and expiry at its authentication boundary, map the principal to the correct tenant, enforce tool permissions on the server, and test concurrent session isolation. Do not use client-provided context values as an authorization source. These are fictional HR policies, not employment advice.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [IBM context-variable tutorial](https://developer.ibm.com/tutorials/context-variables-watsonx-orchestrate/)
- [Official Orchestrate request/response OpenAPI](https://developer.watson-orchestrate.ibm.com/apis/server_openapi.json)
